In [ ]:
import os, glob, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

dataset_dir = "/content/mixed_kashmiri_dataset"


charset_file = "/content/kashmiri_charset.txt"

with open(charset_file, encoding="utf-8") as f:
    CHARS = f.read().strip()

CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}

def text_to_labels(text):
    return [CHAR2IDX.get(c, 0) for c in text]

def labels_to_text(labels):
    return ''.join([IDX2CHAR.get(i, '') for i in labels])

class OCRDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.samples = []
        for img_path in glob.glob(os.path.join(split_dir, "*.png")):
            base = os.path.splitext(os.path.basename(img_path))[0]
            txt_path = os.path.join(split_dir, base + ".txt")
            if os.path.exists(txt_path):
                self.samples.append((img_path, txt_path))
        self.samples.sort()
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, txt_path = self.samples[idx]
        image = Image.open(img_path).convert("L")
        if self.transform:
            image = self.transform(image)
        with open(txt_path, encoding="utf-8") as f:
            label = f.read().strip()
        label_idx = text_to_labels(label)
        return image, torch.tensor(label_idx, dtype=torch.long), len(label_idx)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_outputs):
        attn_weights = torch.softmax(self.attn(rnn_outputs), dim=1)
        context = torch.sum(attn_weights * rnn_outputs, dim=1)
        return context, attn_weights

class CRNN_Attention(nn.Module):
    def __init__(self, img_h, n_channels, n_classes, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(n_channels, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Dropout(dropout)
        )
        self.h_out = img_h // 16
        self.rnn1 = nn.LSTM(512 * self.h_out, 256, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.attn = Attention(512)
        self.fc = nn.Linear(512, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, w, c * h)
        x, _ = self.rnn1(x)
        x = self.dropout(x)
        x, _ = self.rnn2(x)
        x = self.dropout(x)
        context, attn_weights = self.attn(x)
        x = self.fc(x)
        return x.log_softmax(2), attn_weights

def ctc_greedy_decoder(log_probs, blank=0):
    preds = log_probs.argmax(-1)
    pred_texts = []
    for pred in preds.permute(1, 0):
        seq, prev = [], blank
        for p in pred:
            p = p.item()
            if p != prev and p != blank:
                seq.append(p)
            prev = p
        pred_texts.append(seq)
    return pred_texts

def collate_fn(batch):
    images, labels, label_lens = zip(*batch)
    images = torch.stack(images)
    label_lens = torch.tensor(label_lens, dtype=torch.long)
    labels = torch.cat(labels)
    return images, labels, label_lens

def train_crnn(num_epochs=50, batch_size=16, img_h=64, patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose([
        transforms.Resize((img_h, 256)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_ds = OCRDataset(os.path.join(dataset_dir, "train"), transform)
    val_ds   = OCRDataset(os.path.join(dataset_dir, "val"), transform)

    print("Train samples:", len(train_ds))
    print("Val samples:  ", len(val_ds))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    n_classes = len(CHAR2IDX) + 1
    model = CRNN_Attention(img_h, 1, n_classes).to(device)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_val_loss = float("inf")
    patience_counter = 0

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, num_epochs + 1):
        # -------- TRAIN --------
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for images, labels, label_lens in train_loader:
            images, labels, label_lens = images.to(device), labels.to(device), label_lens.to(device)
            optimizer.zero_grad()
            logits, _ = model(images)
            log_probs = logits.permute(1, 0, 2)
            input_lens = torch.full((images.size(0),), logits.size(1), dtype=torch.long, device=device)
            loss = criterion(log_probs, labels, input_lens, label_lens)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            preds = ctc_greedy_decoder(log_probs.detach().cpu())
            offset = 0
            for i, l_len in enumerate(label_lens):
                gt = labels[offset:offset + int(l_len)].cpu().tolist()
                offset += int(l_len)
                if preds[i] == gt:
                    correct += 1
                total += 1

        train_loss /= len(train_loader)
        train_acc = correct / total if total > 0 else 0.0


        model.eval()
        val_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels, label_lens in val_loader:
                images, labels, label_lens = images.to(device), labels.to(device), label_lens.to(device)
                logits, _ = model(images)
                log_probs = logits.permute(1, 0, 2)
                input_lens = torch.full((images.size(0),), logits.size(1), dtype=torch.long, device=device)
                loss = criterion(log_probs, labels, input_lens, label_lens)
                val_loss += loss.item()

                preds = ctc_greedy_decoder(log_probs.cpu())
                offset = 0
                for i, l_len in enumerate(label_lens):
                    gt = labels[offset:offset + int(l_len)].cpu().tolist()
                    offset += int(l_len)
                    if preds[i] == gt:
                        correct += 1
                    total += 1

        val_loss /= len(val_loader)
        val_acc = correct / total if total > 0 else 0.0

        print(f"Epoch {epoch:02d} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "crnn_attention_best.pth")
            print("✅ Model saved")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("⏹ Early stopping")
                break

    print("🎉 Training complete")


if __name__ == "__main__":
    train_crnn()


Train samples: 54720
Val samples:   7816
Epoch 01 | Train Loss: 3.4381 | Val Loss: 2.7198 | Train Acc: 0.0001 | Val Acc: 0.0006
✅ Model saved
Epoch 02 | Train Loss: 2.0924 | Val Loss: 1.3426 | Train Acc: 0.0183 | Val Acc: 0.0778
✅ Model saved
Epoch 03 | Train Loss: 1.1068 | Val Loss: 0.8494 | Train Acc: 0.1573 | Val Acc: 0.2453
✅ Model saved
Epoch 04 | Train Loss: 0.7087 | Val Loss: 0.5592 | Train Acc: 0.3388 | Val Acc: 0.4126
✅ Model saved
Epoch 05 | Train Loss: 0.5535 | Val Loss: 0.4205 | Train Acc: 0.4389 | Val Acc: 0.5256
✅ Model saved
Epoch 06 | Train Loss: 0.4565 | Val Loss: 0.3441 | Train Acc: 0.5202 | Val Acc: 0.6003
✅ Model saved
Epoch 07 | Train Loss: 0.4006 | Val Loss: 0.2881 | Train Acc: 0.5704 | Val Acc: 0.6530
✅ Model saved
Epoch 08 | Train Loss: 0.3864 | Val Loss: 0.2429 | Train Acc: 0.5864 | Val Acc: 0.7117
✅ Model saved
Epoch 09 | Train Loss: 0.3264 | Val Loss: 0.2229 | Train Acc: 0.6329 | Val Acc: 0.7446
✅ Model saved
Epoch 10 | Train Loss: 0.3040 | Val Loss: 0.2373 |

In [1]:
import os, glob, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import editdistance
from collections import defaultdict

DATASET_DIR = "/content/mixed_kashmiri_dataset/test"
MODEL_PATH = "/content/CRNN_BiLSTM_Mixed.pth"
CHARSET_FILE = "/content/kashmiri_charset.txt"

IMG_H = 64
IMG_W = 256
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(CHARSET_FILE, encoding="utf-8") as f:
    CHARS = f.read().strip()

assert len(CHARS) > 1, "❌ Charset is empty"

CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}
N_CLASSES = len(CHAR2IDX) + 1

print("✅ Charset size:", N_CLASSES)

class OCRDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        for img_path in glob.glob(os.path.join(root_dir, "*.png")):
            txt_path = img_path.replace(".png", ".txt")
            if os.path.exists(txt_path):
                self.samples.append((img_path, txt_path))
        self.samples.sort()
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, txt_path = self.samples[idx]
        image = Image.open(img_path).convert("L")
        if self.transform:
            image = self.transform(image)
        with open(txt_path, encoding="utf-8") as f:
            gt = f.read().strip()
        return image, gt

def collate_fn(batch):
    images, texts = zip(*batch)
    return torch.stack(images), texts

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_outputs):
        weights = torch.softmax(self.attn(rnn_outputs), dim=1)
        context = torch.sum(weights * rnn_outputs, dim=1)
        return context, weights

class CRNN_Attention(nn.Module):
    def __init__(self, img_h, n_channels, n_classes, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(n_channels, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d((2,1)),
            nn.Dropout(dropout)
        )

        self.h_out = img_h // 16
        self.rnn1 = nn.LSTM(512 * self.h_out, 256, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(512, 256, bidirectional=True, batch_first=True)
        self.attn = Attention(512)
        self.fc = nn.Linear(512, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, w, c * h)
        x, _ = self.rnn1(x)
        x = self.dropout(x)
        x, _ = self.rnn2(x)
        x = self.dropout(x)
        _, attn_weights = self.attn(x)
        x = self.fc(x)
        return x.log_softmax(2), attn_weights

model = CRNN_Attention(IMG_H, 1, N_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

print("✅ Model loaded successfully")

transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def ctc_greedy_decoder(log_probs, blank=0):
    preds = log_probs.argmax(-1)
    results = []
    for pred in preds.permute(1, 0):
        seq, prev = [], blank
        for p in pred:
            p = p.item()
            if p != prev and p != blank:
                seq.append(p)
            prev = p
        results.append(seq)
    return results

def evaluate_folder(folder):
    ds = OCRDataset(folder, transform)
    if len(ds) == 0:
        return None

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    char_edits = chars = 0
    word_edits = words = 0

    with torch.no_grad():
        for images, gts in loader:
            images = images.to(device)
            logits, _ = model(images)
            log_probs = logits.permute(1, 0, 2)
            preds = ctc_greedy_decoder(log_probs.cpu())

            for seq, gt in zip(preds, gts):
                pred_text = "".join(IDX2CHAR.get(i, "") for i in seq)

                char_edits += editdistance.eval(pred_text, gt)
                chars += len(gt)

                gt_words = gt.split()
                pred_words = pred_text.split()

                if len(gt_words) > 0:
                    word_edits += editdistance.eval(pred_words, gt_words)
                    words += len(gt_words)

    return char_edits, chars, word_edits, words

overall = defaultdict(int)

for font in sorted(os.listdir(DATASET_DIR)):
    font_dir = os.path.join(DATASET_DIR, font)

    for subset in ["clean", "noisy"]:
        res = evaluate_folder(os.path.join(font_dir, subset))

        if res:
            ce, ch, we, wd = res
            overall["char_edits"] += ce
            overall["chars"] += ch
            overall["word_edits"] += we
            overall["words"] += wd

print("\n📊 OVERALL RESULTS (ALL FONTS + CLEAN + NOISY)")
print(f"Overall CER: {overall['char_edits']/overall['chars']:.4f}")
print(f"Overall WER: {overall['word_edits']/overall['words']:.4f}")

✅ Charset size: 74
✅ Model loaded successfully

📊 OVERALL RESULTS (ALL FONTS + CLEAN + NOISY)
Overall CER: 0.0660
Overall WER: 0.2554
